# VoxCPM2 Dual‑T4 Voice Studio — Final UI

Kaggle-focused Burmese voice studio with two T4 workers, long-form chunking, 12 voice-design profiles, persistent saved voices, remote/direct voice cloning, optional cleanup, Whisper transcription, deterministic speed control, and exact chunk-join pauses.

**Important:** restart the Kaggle session once before switching from the older notebook version so old GPU worker processes do not remain in VRAM.


In [1]:
# Cell 1 — install Voice Studio dependencies
!pip -q install -U \
    voxcpm soundfile kagglehub huggingface_hub \
    yt-dlp gdown faster-whisper noisereduce \
    mega.py-v2 demucs

# Kaggle normally has ffmpeg; install it only when missing.
!which ffmpeg >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq ffmpeg)

print("✓ Voice Studio dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 2.5 MB/s eta 0:00:00
   ━━━━

In [2]:
# Cell 2 — download model once to a shared local directory
from huggingface_hub import snapshot_download
import os

MODEL_DIR = "/kaggle/working/VoxCPM2"

if not os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")):
    print("Downloading VoxCPM2...")
    snapshot_download(
        repo_id="openbmb/VoxCPM2",
        local_dir=MODEL_DIR,
    )
else:
    print("✓ VoxCPM2 already exists in", MODEL_DIR)

print("✓ Model directory:", MODEL_DIR)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✓ Model directory: /kaggle/working/VoxCPM2


In [3]:
%%writefile /kaggle/working/voxcpm_dual_worker.py
import os
import random
import inspect
import traceback

def worker_main(physical_gpu_id, model_dir, task_q, result_q, optimize=False):
    # One physical T4 is exposed to each child. Inside the child, "cuda" is correct.
    os.environ["CUDA_VISIBLE_DEVICES"] = str(physical_gpu_id)

    try:
        import numpy as np
        import torch
        import soundfile as sf
        from voxcpm import VoxCPM

        result_q.put({
            "status": "loading",
            "gpu": physical_gpu_id,
        })

        model = VoxCPM.from_pretrained(
            model_dir,
            load_denoiser=False,
            optimize=bool(optimize),
            device="cuda",
        )

        try:
            sample_rate = int(model.tts_model.sample_rate)
        except Exception:
            sample_rate = 48000

        # Detect the parameters accepted by the *installed* VoxCPM build.
        try:
            sig = inspect.signature(model._generate)
            params = sig.parameters
            accepts_any_kwarg = any(
                p.kind == inspect.Parameter.VAR_KEYWORD
                for p in params.values()
            )
            supported = set(params.keys())
        except Exception:
            accepts_any_kwarg = True
            supported = set()

        result_q.put({
            "status": "ready",
            "gpu": physical_gpu_id,
            "sample_rate": sample_rate,
            "supported_params": sorted(supported),
        })

    except Exception as e:
        result_q.put({
            "status": "startup_error",
            "gpu": physical_gpu_id,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        return

    while True:
        task = task_q.get()

        if task is None:
            break

        batch_id = task["batch_id"]
        task_id = task["task_id"]
        chunk_index = task.get("chunk_index", 0)
        output_path = task["output_path"]
        kwargs = dict(task["kwargs"])

        try:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # ---------------------------------------------------------
            # VERSION-COMPATIBILITY LAYER
            # ---------------------------------------------------------
            # Recent VoxCPM builds accept seed= in _generate().
            # The Kaggle build in this user's runtime does not.
            #
            # If seed isn't supported, set Python/NumPy/PyTorch RNGs
            # manually, then REMOVE seed before model.generate().
            # ---------------------------------------------------------
            requested_seed = kwargs.get("seed", None)

            if (
                requested_seed is not None
                and not accepts_any_kwarg
                and "seed" not in supported
            ):
                requested_seed = int(requested_seed)

                random.seed(requested_seed)
                np.random.seed(requested_seed % (2**32 - 1))
                torch.manual_seed(requested_seed)

                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(requested_seed)

                kwargs.pop("seed", None)

            # Filter every unsupported UI option too. This makes the notebook
            # resilient if Kaggle has an older VoxCPM than the current source.
            dropped_kwargs = []

            if not accepts_any_kwarg:
                for key in list(kwargs.keys()):
                    if key not in supported:
                        dropped_kwargs.append(key)
                        kwargs.pop(key, None)

            wav = model.generate(**kwargs)

            sf.write(
                output_path,
                wav,
                sample_rate,
                subtype="PCM_16",
            )

            result_q.put({
                "status": "ok",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "output_path": output_path,
                "sample_rate": sample_rate,
                "samples": int(len(wav)),
                "duration": float(len(wav) / sample_rate),
                "dropped_kwargs": dropped_kwargs,
            })

        except Exception as e:
            result_q.put({
                "status": "error",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })


Writing /kaggle/working/voxcpm_dual_worker.py


In [4]:
# Cell 4 — dual-GPU service + persistent voice library + UI
import os
import base64
import re
import sys
import json
import glob
import time
import uuid
import shutil
import queue as pyqueue
from pathlib import Path
from multiprocessing import get_context

import numpy as np
import soundfile as sf
import ipywidgets as widgets

from IPython.display import display, Audio, FileLink, clear_output, HTML, Javascript

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

MODEL_DIR = "/kaggle/working/VoxCPM2"

# Change this only if your Kaggle dataset has a different owner/slug.
# This is used by the optional "Sync library to Kaggle Dataset" button.
VOICE_DATASET_HANDLE_DEFAULT = "cyawzin/voxcpm-voice-library"

# T4-safe default. Set True later only if you want to benchmark torch.compile.
# optimize=True can take a long time to compile/warm up on T4.
WORKER_OPTIMIZE = False

# Allow both child processes plenty of time to load the model.
WORKER_STARTUP_TIMEOUT = 900  # 15 minutes

LIBRARY_ROOT = Path("/kaggle/working/voxcpm_voice_library")
VOICE_DIR = LIBRARY_ROOT / "voices"
GENERATED_DIR = LIBRARY_ROOT / "generated"
META_FILE = LIBRARY_ROOT / "library.json"

VOICE_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RESTORE FROM AN ATTACHED KAGGLE DATASET
# ------------------------------------------------------------

def restore_attached_library_if_needed():
    if META_FILE.exists():
        return None

    candidates = glob.glob(
        "/kaggle/input/**/library.json",
        recursive=True,
    )

    for meta in candidates:
        src_root = Path(meta).parent
        src_voice_dir = src_root / "voices"

        if src_voice_dir.exists():
            shutil.copytree(
                src_root,
                LIBRARY_ROOT,
                dirs_exist_ok=True,
            )
            return str(src_root)

    return None

restored_from = restore_attached_library_if_needed()

# ------------------------------------------------------------
# LIBRARY HELPERS
# ------------------------------------------------------------

def empty_library():
    return {
        "version": 2,
        "active_voice_id": None,
        "voices": {},
    }

def load_library():
    if not META_FILE.exists():
        return empty_library()

    try:
        with open(META_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        data.setdefault("version", 2)
        data.setdefault("active_voice_id", None)
        data.setdefault("voices", {})
        return data
    except Exception:
        return empty_library()

library = load_library()

def save_library():
    LIBRARY_ROOT.mkdir(parents=True, exist_ok=True)
    with open(META_FILE, "w", encoding="utf-8") as f:
        json.dump(
            library,
            f,
            ensure_ascii=False,
            indent=2,
        )

def voice_path(voice_id):
    entry = library["voices"][voice_id]
    return LIBRARY_ROOT / entry["file"]

def valid_voice_ids():
    out = []
    for voice_id, entry in library["voices"].items():
        p = LIBRARY_ROOT / entry["file"]
        if p.exists():
            out.append(voice_id)
    return out

# ------------------------------------------------------------
# LONG-TEXT CHUNKING
# ------------------------------------------------------------

BOUNDARIES = ["။", "!", "?", "…", ".", "၊", ",", ";", ":", " "]

def split_long_text(text, max_chars=220):
    """
    Split at natural Burmese/English boundaries.
    max_chars is a safety target, not an audio-duration guarantee.
    """
    text = re.sub(r"[ \t]+", " ", text or "")
    text = re.sub(r"\n+", "\n", text).strip()

    if not text:
        return []

    chunks = []
    remaining = text

    while len(remaining) > max_chars:
        window = remaining[:max_chars + 1]

        best = -1
        for mark in BOUNDARIES:
            pos = window.rfind(mark)
            if pos > best:
                best = pos

        # Avoid creating a tiny chunk just because there was old punctuation.
        if best < int(max_chars * 0.55):
            best = max_chars
        else:
            best += 1

        piece = remaining[:best].strip()
        if piece:
            chunks.append(piece)

        remaining = remaining[best:].strip()

    if remaining:
        chunks.append(remaining)

    return chunks

def concatenate_wavs(paths, output_path, pause_ms=90):
    arrays = []
    target_sr = None

    for i, p in enumerate(paths):
        wav, sr = sf.read(p, dtype="float32", always_2d=False)

        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        if target_sr is None:
            target_sr = sr
        elif sr != target_sr:
            raise RuntimeError(
                f"Sample-rate mismatch: expected {target_sr}, got {sr}"
            )

        arrays.append(wav)

        if i != len(paths) - 1 and pause_ms > 0:
            arrays.append(
                np.zeros(
                    int(target_sr * pause_ms / 1000.0),
                    dtype=np.float32,
                )
            )

    final = np.concatenate(arrays) if arrays else np.zeros(0, dtype=np.float32)

    sf.write(
        output_path,
        final,
        target_sr or 48000,
        subtype="PCM_16",
    )

    return output_path, target_sr or 48000, len(final)

# ------------------------------------------------------------
# TWO PERSISTENT GPU WORKERS
# ------------------------------------------------------------

sys.path.insert(0, "/kaggle/working")
from voxcpm_dual_worker import worker_main

class DualGPUVoxCPM:
    def __init__(
        self,
        model_dir,
        gpu_ids=(0, 1),
        optimize=False,
        startup_timeout=900,
        heartbeat_seconds=10,
    ):
        self.model_dir = model_dir
        self.gpu_ids = tuple(gpu_ids)
        self.ctx = get_context("spawn")
        self.task_q = self.ctx.Queue()
        self.result_q = self.ctx.Queue()
        self.procs = []
        self.sample_rate = 48000

        print(
            f"Starting {len(self.gpu_ids)} VoxCPM2 workers "
            f"(optimize={optimize})..."
        )

        for gpu_id in self.gpu_ids:
            p = self.ctx.Process(
                target=worker_main,
                args=(
                    gpu_id,
                    self.model_dir,
                    self.task_q,
                    self.result_q,
                    optimize,
                ),
                name=f"VoxCPM2-GPU-{gpu_id}",
            )
            p.start()
            self.procs.append(p)
            print(
                f"  • launched worker PID {p.pid} "
                f"for physical GPU {gpu_id}"
            )

        ready = set()
        deadline = time.time() + float(startup_timeout)
        last_status = 0.0

        while len(ready) < len(self.gpu_ids):
            now = time.time()

            for gpu_id, proc in zip(self.gpu_ids, self.procs):
                if gpu_id not in ready and proc.exitcode is not None:
                    self.shutdown(force=True)
                    raise RuntimeError(
                        f"VoxCPM2 worker for GPU {gpu_id} exited during startup "
                        f"with exit code {proc.exitcode}.\n"
                        "Scroll upward for the child-process traceback. "
                        "If this was an OOM kill, restart the Kaggle session "
                        "before running this cell again."
                    )

            if now > deadline:
                statuses = [
                    (
                        f"GPU {gpu}: pid={proc.pid}, "
                        f"alive={proc.is_alive()}, "
                        f"exitcode={proc.exitcode}, "
                        f"ready={gpu in ready}"
                    )
                    for gpu, proc in zip(self.gpu_ids, self.procs)
                ]
                self.shutdown(force=True)
                raise TimeoutError(
                    "Timed out while loading the VoxCPM2 GPU workers.\n"
                    + "\n".join(statuses)
                )

            # FIX: empty queue is normal while a large model is still loading.
            try:
                msg = self.result_q.get(timeout=5)
            except pyqueue.Empty:
                if now - last_status >= heartbeat_seconds:
                    waiting = [
                        str(gpu)
                        for gpu in self.gpu_ids
                        if gpu not in ready
                    ]
                    elapsed = int(
                        startup_timeout - max(0.0, deadline - now)
                    )
                    print(
                        f"  … still loading GPU worker(s) "
                        f"{', '.join(waiting)} "
                        f"({elapsed}s elapsed)"
                    )
                    last_status = now
                continue

            status = msg.get("status")

            if status == "ready":
                gpu = msg["gpu"]
                if gpu not in ready:
                    ready.add(gpu)
                    self.sample_rate = msg.get(
                        "sample_rate",
                        self.sample_rate,
                    )
                    print(
                        f"✓ GPU {gpu} worker ready "
                        f"({self.sample_rate} Hz)"
                    )

            elif status == "startup_error":
                self.shutdown(force=True)
                raise RuntimeError(
                    f"GPU {msg.get('gpu')} failed to start:\n"
                    f"{msg.get('error')}\n\n"
                    f"{msg.get('traceback', '')}"
                )

            else:
                print("Worker startup message:", msg)

        print("✓ All VoxCPM2 GPU workers are ready.")

    def run(self, task_specs, on_result=None):
        """
        Run tasks across both GPU workers.

        on_result(message, finished_count, total_count) fires immediately
        whenever a chunk finishes, which powers progressive playback.
        """
        if not task_specs:
            return []

        batch_id = uuid.uuid4().hex
        wanted = set()

        for i, spec in enumerate(task_specs):
            task_id = f"{batch_id}_{i}"
            wanted.add(task_id)

            payload = dict(spec)
            payload["batch_id"] = batch_id
            payload["task_id"] = task_id
            self.task_q.put(payload)

        results = []
        total = len(task_specs)

        while len(results) < total:
            for gpu_id, proc in zip(self.gpu_ids, self.procs):
                if proc.exitcode is not None:
                    raise RuntimeError(
                        f"GPU worker {gpu_id} died during generation "
                        f"with exit code {proc.exitcode}."
                    )

            try:
                msg = self.result_q.get(timeout=10)
            except pyqueue.Empty:
                continue

            if msg.get("batch_id") != batch_id:
                continue

            if msg.get("task_id") not in wanted:
                continue

            if msg.get("status") != "ok":
                raise RuntimeError(
                    f"GPU {msg.get('gpu')} generation failed:\n"
                    f"{msg.get('error')}\n\n"
                    f"{msg.get('traceback', '')}"
                )

            results.append(msg)

            if on_result is not None:
                try:
                    on_result(msg, len(results), total)
                except Exception as callback_error:
                    # A preview/frontend failure must not destroy synthesis.
                    print(
                        "⚠ Live-preview callback error:",
                        repr(callback_error),
                    )

        return sorted(
            results,
            key=lambda r: r.get("chunk_index", 0),
        )

    def shutdown(self, force=False):
        for _ in self.procs:
            try:
                self.task_q.put_nowait(None)
            except Exception:
                pass

        for p in self.procs:
            try:
                p.join(timeout=5)
            except Exception:
                pass

            if force or p.is_alive():
                try:
                    p.terminate()
                except Exception:
                    pass

# Shut down old workers if this cell is run again.
if "_VOX_SERVICE" in globals():
    try:
        _VOX_SERVICE.shutdown(force=True)
    except Exception:
        pass

print("Loading one VoxCPM2 replica on each T4...")

# IMPORTANT:
# If the previous broken cell already launched workers, restart the Kaggle
# session once before running this replacement cell so no orphan model remains.
#
# The notebook parent intentionally does NOT load VoxCPM2 itself.
# Only the two child workers own model replicas: one per physical T4.

_VOX_SERVICE = DualGPUVoxCPM(
    MODEL_DIR,
    gpu_ids=(0, 1),
    optimize=WORKER_OPTIMIZE,
    startup_timeout=WORKER_STARTUP_TIMEOUT,
    heartbeat_seconds=10,
)

print("✓ Dual-GPU VoxCPM2 service ready")
print("✓ Runtime compatibility filter enabled for older/newer VoxCPM _generate signatures")


# ------------------------------------------------------------
# LIVE / PROGRESSIVE AUDIO HELPERS
# ------------------------------------------------------------

def prepare_stream_chunk(
    raw_path,
    output_path,
    speed=1.0,
    trim_edges=True,
    trim_db=-42.0,
):
    """
    Prepare one finished VoxCPM chunk exactly as it will sound in the
    final combined file. This lets the UI expose it immediately.
    """
    raw_path = Path(raw_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    working = raw_path
    temp_speed = None

    try:
        if abs(float(speed) - 1.0) >= 0.001:
            temp_speed = output_path.with_name(
                output_path.stem + "_speedtmp.wav"
            )
            ffmpeg_atempo(
                str(raw_path),
                str(temp_speed),
                float(speed),
            )
            working = temp_speed

        wav, sr = sf.read(
            str(working),
            dtype="float32",
            always_2d=False,
        )
        wav = _mono_float32(wav)

        if trim_edges:
            wav = trim_edge_silence(
                wav,
                sr,
                threshold_db=float(trim_db),
                keep_ms=25,
            )

        sf.write(
            str(output_path),
            wav,
            sr,
            subtype="PCM_16",
        )

        return str(output_path), int(sr), int(len(wav))

    finally:
        if temp_speed is not None:
            try:
                Path(temp_speed).unlink(missing_ok=True)
            except Exception:
                pass


def concatenate_prepared_chunks(
    prepared_paths,
    output_path,
    pause_ms=120,
):
    """
    Join already-speed-adjusted / trimmed chunks and insert an exact pause.
    """
    arrays = []
    target_sr = None

    for i, path in enumerate(prepared_paths):
        wav, sr = sf.read(
            str(path),
            dtype="float32",
            always_2d=False,
        )
        wav = _mono_float32(wav)

        if target_sr is None:
            target_sr = int(sr)
        elif int(sr) != target_sr:
            raise RuntimeError(
                f"Sample-rate mismatch: {sr} != {target_sr}"
            )

        arrays.append(wav)

        if i != len(prepared_paths) - 1 and int(pause_ms) > 0:
            arrays.append(
                np.zeros(
                    int(target_sr * int(pause_ms) / 1000.0),
                    dtype=np.float32,
                )
            )

    if not arrays:
        raise RuntimeError("No prepared chunks are available.")

    final = np.concatenate(arrays)

    sf.write(
        str(output_path),
        final,
        target_sr or 48000,
        subtype="PCM_16",
    )

    return (
        str(output_path),
        target_sr or 48000,
        int(len(final)),
    )


def make_stream_preview_mp3(
    prepared_paths,
    preview_wav,
    preview_mp3,
    pause_ms,
):
    """
    Build the currently available continuous prefix and encode it to a
    small MP3. The MP3 is sent to the browser player as a data URL.

    MP3 is used only for live preview; the final downloadable file remains WAV.
    """
    concatenate_prepared_chunks(
        prepared_paths,
        preview_wav,
        pause_ms=pause_ms,
    )

    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-i", str(preview_wav),
        "-vn",
        "-ac", "1",
        "-codec:a", "libmp3lame",
        "-b:a", "64k",
        str(preview_mp3),
    ]
    subprocess.run(cmd, check=True)

    return str(preview_mp3)


def file_as_data_url(path, mime="audio/mpeg"):
    encoded = base64.b64encode(
        Path(path).read_bytes()
    ).decode("ascii")
    return f"data:{mime};base64,{encoded}"


# ------------------------------------------------------------
# KAGGLE DATASET SYNC
# ------------------------------------------------------------

def sync_library_to_dataset(handle):
    handle = (handle or "").strip()

    if "/" not in handle:
        raise ValueError(
            "Dataset handle must look like: username/dataset-slug"
        )

    import kagglehub

    # Upload ONLY permanent voice metadata + reference WAVs.
    # Generated output, imported media and cookies are intentionally excluded.
    sync_dir = Path("/kaggle/working/voxcpm_voice_dataset_sync")
    shutil.rmtree(sync_dir, ignore_errors=True)
    (sync_dir / "voices").mkdir(parents=True, exist_ok=True)

    if META_FILE.exists():
        shutil.copy2(META_FILE, sync_dir / "library.json")

    if VOICE_DIR.exists():
        shutil.copytree(VOICE_DIR, sync_dir / "voices", dirs_exist_ok=True)

    kagglehub.dataset_upload(
        handle,
        str(sync_dir),
        version_notes=time.strftime(
            "Voice library sync %Y-%m-%d %H:%M:%S"
        ),
    )

# ------------------------------------------------------------
# FINAL ELEVENLABS-LIKE UI + AUDIO/CLONE HELPERS
# ------------------------------------------------------------

import gc
import threading
import subprocess
from urllib.parse import urlparse

IMPORT_DIR = Path("/kaggle/working/voxcpm_clone_imports")
COOKIE_DIR = Path("/kaggle/working/voxcpm_private_cookies")
IMPORT_DIR.mkdir(parents=True, exist_ok=True)
COOKIE_DIR.mkdir(parents=True, exist_ok=True)

# 12 profiles adapted from the original Character Voice Studio.
VOICE_PROFILES = {
    "u_aung": {
        "name": "👨 U Aung — Deep Calm Male",
        "short": "Deep • calm • mature",
        "design": (
            "An adult Burmese man in his late thirties to mid forties. A naturally low baritone voice with warm chest resonance, "
            "full but relaxed tone, calm confidence and mature presence. Speak in authentic conversational Burmese with clean articulation, "
            "realistic micro-pauses and subtle breathing. Use a relaxed medium-slow pace. Never sound robotic, theatrical or like an advertisement."
        ),
    },
    "ko_min": {
        "name": "👨 Ko Min — Friendly Young Male",
        "short": "Young • friendly • conversational",
        "design": (
            "A Burmese man in his mid to late twenties with a naturally friendly, approachable medium-pitched voice. Warm, relaxed and youthful "
            "without sounding childish. Use everyday Burmese rhythm, smooth phrasing, clear consonants and a comfortable medium pace. "
            "Sound like a real person speaking directly to a friend."
        ),
    },
    "u_htet": {
        "name": "👴 U Htet — Wise Older Male",
        "short": "Older • warm • storyteller",
        "design": (
            "A Burmese man around sixty with a mature naturally deep voice, gentle age texture and warmth. Use a wise, patient, reassuring storyteller "
            "delivery with measured pacing and thoughtful pauses. Keep pronunciation natural and distinctly Burmese. Avoid exaggerated elderly acting or raspiness."
        ),
    },
    "ko_zaw": {
        "name": "👨 Ko Zaw — Energetic Presenter",
        "short": "Energetic • clear • engaging",
        "design": (
            "A Burmese man around thirty with a clear, energetic and confident speaking voice. Sound naturally enthusiastic rather than loud. "
            "Use lively but controlled pitch movement, crisp articulation, smooth connected speech and slightly quick pacing while keeping every word clear."
        ),
    },
    "u_kyaw": {
        "name": "👔 U Kyaw — Formal Professional Male",
        "short": "Formal • polished • trustworthy",
        "design": (
            "A professional Burmese man in his forties with a composed, polished, trustworthy voice. Use precise articulation, stable volume, confident phrasing "
            "and a measured medium pace. Formal enough for business or educational narration, but still warm and human rather than stiff."
        ),
    },
    "daw_may": {
        "name": "👩 Daw May — Warm Adult Female",
        "short": "Warm • caring • grounded",
        "design": (
            "A Burmese woman in her late thirties to mid forties with a warm, gentle, naturally feminine voice. Caring, grounded and trustworthy with soft but clear "
            "articulation, smooth sentence transitions, realistic breathing and a relaxed medium pace. Avoid an overly sweet or commercial sound."
        ),
    },
    "ma_su": {
        "name": "👩 Ma Su — Bright Young Female",
        "short": "Young • bright • friendly",
        "design": (
            "A Burmese woman in her early to mid twenties with a youthful, bright and naturally pleasant voice. Cheerful, friendly and expressive without becoming childish. "
            "Use light natural energy, clear pronunciation, smooth connected speech and realistic conversational pitch movement."
        ),
    },
    "daw_thida": {
        "name": "👩 Daw Thida — Elegant Mature Female",
        "short": "Elegant • mature • composed",
        "design": (
            "A mature Burmese woman with a smooth, composed and elegant voice. Refined, confident, warm and professionally controlled without becoming distant. "
            "Use excellent articulation, balanced resonance, natural Burmese phrasing, subtle pitch variation and a measured medium pace."
        ),
    },
    "ma_nandar": {
        "name": "👩 Ma Nandar — Soft Emotional Female",
        "short": "Soft • intimate • emotional",
        "design": (
            "A Burmese woman in her late twenties to early thirties with a soft, intimate and emotionally expressive voice. Close, personal and sincere while remaining clearly audible. "
            "Use gentle breath control, subtle changes in pitch and intensity, soft sentence endings and realistic pauses. Avoid melodrama or excessive whispering."
        ),
    },
    "news_presenter": {
        "name": "📰 Burmese News Presenter",
        "short": "Neutral • formal • broadcast",
        "design": (
            "A professional adult Burmese news presenter with a neutral, trustworthy and highly intelligible broadcast voice. Use balanced resonance, precise pronunciation, stable volume "
            "and controlled sentence emphasis. Maintain a steady medium pace and restrained natural prosody rather than flat robotic reading."
        ),
    },
    "documentary": {
        "name": "🎥 Deep Documentary Narrator",
        "short": "Deep • cinematic • serious",
        "design": (
            "A mature Burmese documentary narrator with a naturally deep, resonant voice. Intelligent, observant, serious and quietly cinematic. Use controlled low-register resonance, "
            "deliberate medium-slow pacing, excellent clarity, thoughtful pauses, subtle suspense and restrained emotion."
        ),
    },
    "movie_recap": {
        "name": "🎬 Fast Movie Recap Narrator",
        "short": "Fast • exciting • high retention",
        "design": (
            "A young adult Burmese male movie-recap narrator with a natural, energetic and highly engaging voice. Speak noticeably faster than normal conversation while preserving clean pronunciation "
            "and intelligibility. Maintain strong forward momentum, short pauses, quick transitions, controlled excitement, curiosity and tension. Never shout or slur words."
        ),
    },
}

DELIVERY_PRESETS = {
    "Natural / keep cloned delivery": "",
    "Warm & conversational": "warm, natural, conversational delivery with subtle human pauses",
    "Calm & serious": "calm, serious, controlled delivery with clear articulation",
    "Friendly": "friendly, approachable, relaxed conversational delivery",
    "Professional / formal": "professional, formal, polished delivery with precise articulation",
    "Documentary": "deep, restrained documentary narration with thoughtful pacing and subtle suspense",
    "Movie recap": "fast-paced engaging movie recap delivery with strong momentum and clean articulation",
    "Excited": "energetic and excited but controlled, clear and never shouting",
    "Soft / emotional": "soft, intimate and sincere delivery with gentle emotional variation",
}

# ---------------------------
# Enhanced final-audio tools
# ---------------------------

def _mono_float32(wav):
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    return wav


def trim_edge_silence(wav, sr, threshold_db=-42.0, keep_ms=25):
    """Trim leading/trailing silence only; never removes interior pauses."""
    wav = _mono_float32(wav)
    if len(wav) == 0:
        return wav
    peak = float(np.max(np.abs(wav)))
    if peak < 1e-7:
        return wav
    threshold = peak * (10.0 ** (float(threshold_db) / 20.0))
    active = np.flatnonzero(np.abs(wav) >= threshold)
    if active.size == 0:
        return wav
    keep = int(sr * float(keep_ms) / 1000.0)
    start = max(0, int(active[0]) - keep)
    end = min(len(wav), int(active[-1]) + keep + 1)
    return wav[start:end]


def ffmpeg_atempo(input_path, output_path, speed):
    speed = float(speed)
    if abs(speed - 1.0) < 0.001:
        shutil.copy2(input_path, output_path)
        return str(output_path)
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(input_path),
        "-filter:a", f"atempo={speed:.4f}",
        "-c:a", "pcm_s16le",
        str(output_path),
    ]
    subprocess.run(cmd, check=True)
    return str(output_path)


def concatenate_wavs(paths, output_path, pause_ms=120, trim_edges=True, trim_db=-42.0, speed=1.0):
    """
    Speed is applied per chunk BEFORE joining. Then edge silence is trimmed and
    the exact requested silence is inserted, so 100 ms really means ~100 ms.
    """
    processed = []
    temp_dir = Path(output_path).parent / ("_post_" + uuid.uuid4().hex[:8])
    temp_dir.mkdir(parents=True, exist_ok=True)
    try:
        for i, p in enumerate(paths):
            source = Path(p)
            if abs(float(speed) - 1.0) >= 0.001:
                sped = temp_dir / f"speed_{i:04d}.wav"
                ffmpeg_atempo(source, sped, speed)
                source = sped

            wav, sr = sf.read(str(source), dtype="float32", always_2d=False)
            wav = _mono_float32(wav)
            if trim_edges:
                wav = trim_edge_silence(wav, sr, threshold_db=trim_db, keep_ms=25)
            processed.append((wav, sr))

        if not processed:
            raise RuntimeError("No generated chunks to join.")

        target_sr = processed[0][1]
        arrays = []
        for i, (wav, sr) in enumerate(processed):
            if sr != target_sr:
                raise RuntimeError(f"Sample-rate mismatch: {sr} != {target_sr}")
            arrays.append(wav)
            if i != len(processed) - 1 and int(pause_ms) > 0:
                arrays.append(np.zeros(int(target_sr * int(pause_ms) / 1000.0), dtype=np.float32))

        final = np.concatenate(arrays)
        sf.write(str(output_path), final, target_sr, subtype="PCM_16")
        return str(output_path), target_sr, len(final)
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)


# ---------------------------
# Clone media + ASR helpers
# ---------------------------

def _first_upload(widget):
    value = widget.value
    if not value:
        return None
    if isinstance(value, (tuple, list)):
        item = value[0]
        return {"name": item.get("name", "upload.bin"), "content": bytes(item["content"])}
    if isinstance(value, dict):
        first_key = next(iter(value))
        item = value[first_key]
        name = item.get("metadata", {}).get("name", first_key)
        return {"name": name, "content": bytes(item["content"])}
    return None


def save_upload_to_dir(upload_widget, dest_dir, fallback_name="upload.bin"):
    item = _first_upload(upload_widget)
    if item is None:
        raise ValueError("No local file has been uploaded.")
    name = Path(item["name"] or fallback_name).name
    dest = Path(dest_dir) / name
    dest.write_bytes(item["content"])
    return str(dest)


def save_cookie_upload(cookie_widget):
    item = _first_upload(cookie_widget)
    if item is None:
        return None
    path = COOKIE_DIR / "cookies.txt"
    path.write_bytes(item["content"])
    return str(path)


def normalize_media_to_wav(source_path, output_path, start_seconds=0.0, max_seconds=20.0):
    cmd = ["ffmpeg", "-y", "-loglevel", "error"]
    if float(start_seconds) > 0:
        cmd += ["-ss", str(float(start_seconds))]
    cmd += ["-i", str(source_path)]
    if float(max_seconds) > 0:
        cmd += ["-t", str(float(max_seconds))]
    cmd += ["-vn", "-ac", "1", "-ar", "48000", "-c:a", "pcm_s16le", str(output_path)]
    subprocess.run(cmd, check=True)
    return str(output_path)


def reduce_residual_noise(input_path, output_path, amount=0.70):
    import noisereduce as nr
    wav, sr = sf.read(str(input_path), dtype="float32", always_2d=False)
    wav = _mono_float32(wav)
    cleaned = nr.reduce_noise(y=wav, sr=sr, stationary=False, prop_decrease=float(amount))
    sf.write(str(output_path), cleaned, sr, subtype="PCM_16")
    return str(output_path)


def extract_vocals_demucs(input_path, job_dir):
    # CPU intentionally: both T4s stay dedicated to VoxCPM2.
    out_root = Path(job_dir) / "demucs"
    cmd = [
        sys.executable, "-m", "demucs.separate",
        "-d", "cpu", "-n", "htdemucs",
        "--two-stems=vocals", "--float32",
        "--shifts", "0", "--overlap", "0.10",
        "-o", str(out_root), str(input_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    candidates = list(out_root.glob("**/vocals.wav"))
    if not candidates:
        raise RuntimeError("Demucs completed but vocals.wav was not found.")
    return str(candidates[0])


def transcribe_reference(wav_path, model_name="large-v3-turbo", language="my"):
    from faster_whisper import WhisperModel
    model = WhisperModel(
        model_name,
        device="cpu",
        compute_type="int8",
        cpu_threads=max(2, min(8, os.cpu_count() or 4)),
    )
    lang = None if language == "auto" else language
    segments, info = model.transcribe(
        str(wav_path), language=lang, task="transcribe", beam_size=5,
        vad_filter=True, condition_on_previous_text=True,
    )
    parts = []
    for seg in segments:
        t = (seg.text or "").strip()
        if t:
            parts.append(t)
    detected = getattr(info, "language", None)
    del model
    gc.collect()
    return " ".join(parts).strip(), detected


def download_remote_media(url, job_dir, cookie_file=None, user_agent="", referer=""):
    url = (url or "").strip()
    if not url:
        raise ValueError("Remote URL is empty.")

    host = (urlparse(url).netloc or "").lower()
    job_dir = Path(job_dir)
    job_dir.mkdir(parents=True, exist_ok=True)

    if "drive.google.com" in host or "docs.google.com" in host:
        import gdown
        out = job_dir / "drive_source"
        result = gdown.download(url=url, output=str(out), quiet=False, fuzzy=True)
        if not result:
            raise RuntimeError("Google Drive download failed.")
        return str(result)

    if "mega.nz" in host or "mega.co.nz" in host:
        from mega import Mega
        mega = Mega()
        client = mega.login()
        before = set(job_dir.iterdir())
        result = client.download_url(url, dest_path=str(job_dir))
        if result and Path(result).exists():
            return str(result)
        after = set(job_dir.iterdir())
        new_files = [p for p in (after - before) if p.is_file()]
        if not new_files:
            raise RuntimeError("MEGA download did not create a file.")
        return str(max(new_files, key=lambda p: p.stat().st_size))

    import yt_dlp
    outtmpl = str(job_dir / "remote_source.%(ext)s")
    opts = {
        "format": "bestaudio/best",
        "outtmpl": outtmpl,
        "noplaylist": True,
        "quiet": True,
        "retries": 3,
        "fragment_retries": 3,
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    }
    if cookie_file and Path(cookie_file).exists():
        opts["cookiefile"] = str(cookie_file)
    headers = {}
    if user_agent.strip():
        headers["User-Agent"] = user_agent.strip()
    if referer.strip():
        headers["Referer"] = referer.strip()
    if headers:
        opts["http_headers"] = headers

    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

    candidates = [p for p in job_dir.glob("remote_source.*") if p.is_file() and p.suffix != ".part"]
    if not candidates:
        raise RuntimeError("Remote download did not produce a media file.")
    return str(max(candidates, key=lambda p: p.stat().st_size))


# ---------------------------
# UI state + CSS
# ---------------------------

STATE = {
    "audition_path": None,
    "audition_text": None,
    "audition_direction": None,
    "audition_profile_id": None,
    "clone_path": None,
    "clone_transcript": "",
    "clone_source": "",
    "clone_busy": False,
    "audition_busy": False,
    "generation_busy": False,
    "cookie_file": None,
}

studio_css = widgets.HTML(r"""
<style>
.vox-card {background:#181818;border:1px solid #343434;border-radius:14px;padding:16px 18px;margin:8px 0 14px 0;}
.vox-title {font-size:27px;font-weight:760;color:#fff;}
.vox-sub {color:#aaa;font-size:13px;margin-top:5px;}
.vox-section {color:#fff;font-size:18px;font-weight:700;margin:10px 0 8px 0;}
.vox-note {color:#bdbdbd;font-size:12px;line-height:1.55;}
.widget-button {border-radius:10px!important;font-weight:650!important;}
.widget-textarea textarea,.widget-text input,.widget-dropdown select {border-radius:9px!important;}
</style>
""")

header = widgets.HTML(r"""
<div class="vox-card">
  <div class="vox-title">🎙️ VoxCPM2 Voice Studio</div>
  <div class="vox-sub">Dual T4 • Burmese long-form • voice design • cloning • persistent library</div>
</div>
""")

# ---------------------------
# Shared voice selector
# ---------------------------

saved_voice = widgets.Dropdown(description="Voice", layout=widgets.Layout(width="520px"))
play_selected_btn = widgets.Button(description="▶ Play", layout=widgets.Layout(width="95px"))
delete_selected_btn = widgets.Button(description="🗑 Delete", button_style="danger", layout=widgets.Layout(width="110px"))
library_output = widgets.Output()


def selected_voice_id():
    return saved_voice.value or ""


def refresh_saved_voices(select_id=None):
    ids = valid_voice_ids()
    ids.sort(key=lambda vid: library["voices"][vid].get("created", ""), reverse=True)
    if ids:
        saved_voice.options = [
            (f"{library['voices'][vid].get('name', vid)}   •   {library['voices'][vid].get('created', '')}", vid)
            for vid in ids
        ]
        desired = select_id or library.get("active_voice_id") or ids[0]
        saved_voice.value = desired if desired in ids else ids[0]
        play_selected_btn.disabled = False
        delete_selected_btn.disabled = False
    else:
        saved_voice.options = [("— no saved voice yet —", "")]
        saved_voice.value = ""
        play_selected_btn.disabled = True
        delete_selected_btn.disabled = True


def on_voice_change(change=None):
    vid = selected_voice_id()
    if vid and vid in library["voices"]:
        library["active_voice_id"] = vid
        save_library()

saved_voice.observe(on_voice_change, names="value")


def on_play_selected(_):
    library_output.clear_output(wait=True)

    vid = selected_voice_id()

    if not vid:
        library_output.append_stdout("No voice selected.\n")
        return

    entry = library["voices"][vid]
    path = voice_path(vid)

    library_output.append_stdout(
        f"{entry.get('name', vid)}\n"
        f"Source: {entry.get('source_type', 'saved voice')}\n\n"
    )

    library_output.append_display_data(
        Audio(
            filename=str(path),
            autoplay=False,
        )
    )


def on_delete_selected(_):
    with library_output:
        clear_output()
        if STATE["generation_busy"]:
            print("A generation is running. Delete the voice after it finishes.")
            return
        vid = selected_voice_id()
        if not vid:
            return
        entry = library["voices"].get(vid, {})
        p = LIBRARY_ROOT / entry.get("file", "")
        try:
            if p.exists():
                p.unlink()
        except Exception:
            pass
        library["voices"].pop(vid, None)
        if library.get("active_voice_id") == vid:
            library["active_voice_id"] = None
        save_library()
        refresh_saved_voices()
        print("✓ Voice deleted from the working library.")

play_selected_btn.on_click(on_play_selected)
delete_selected_btn.on_click(on_delete_selected)


# ---------------------------
# Generate tab
# ---------------------------

gen_script = widgets.Textarea(value="", placeholder="Paste your Burmese or English script here...", layout=widgets.Layout(width="100%", height="290px"))
delivery_preset = widgets.Dropdown(options=list(DELIVERY_PRESETS.keys()), value="Natural / keep cloned delivery", description="Delivery", layout=widgets.Layout(width="520px"))
custom_style = widgets.Text(value="", description="Custom", placeholder="emotion/tone only — e.g. calm, tense, cheerful", layout=widgets.Layout(width="100%"))
speed_slider = widgets.FloatSlider(value=1.00, min=0.75, max=1.50, step=0.05, description="Speed", readout_format=".2f", continuous_update=False, layout=widgets.Layout(width="520px"))
chunk_chars = widgets.IntSlider(value=220, min=120, max=360, step=10, description="Chunk chars", continuous_update=False, layout=widgets.Layout(width="520px"))
join_pause = widgets.IntSlider(value=120, min=0, max=1000, step=20, description="Join pause", continuous_update=False, layout=widgets.Layout(width="520px"))
trim_edges_checkbox = widgets.Checkbox(value=True, description="Trim generated edge silence before joining", indent=False)
trim_db = widgets.FloatSlider(value=-42.0, min=-60.0, max=-25.0, step=1.0, description="Silence dB", continuous_update=False, layout=widgets.Layout(width="520px"))
cfg_slider = widgets.FloatSlider(value=2.0, min=1.0, max=3.0, step=0.1, description="CFG", continuous_update=False, layout=widgets.Layout(width="520px"))
steps_slider = widgets.IntSlider(value=10, min=4, max=30, step=1, description="Steps", continuous_update=False, layout=widgets.Layout(width="520px"))
seed_box = widgets.IntText(value=0, description="Seed", layout=widgets.Layout(width="260px"))
random_seed_checkbox = widgets.Checkbox(value=True, description="Random seed", indent=False)
generate_btn = widgets.Button(description="⚡ Generate + Stream", button_style="success", layout=widgets.Layout(width="280px", height="50px"))
gen_status = widgets.HTML()
# One stable browser player. Its source is replaced with a longer cumulative
# MP3 after each contiguous chunk becomes available.
STREAM_PLAYER_ID = "voxcpm_stream_" + uuid.uuid4().hex[:10]
STREAM_STATUS_ID = STREAM_PLAYER_ID + "_status"

gen_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="Progress",
    bar_style="info",
    orientation="horizontal",
    layout=widgets.Layout(width="100%"),
)

gen_stream_summary = widgets.HTML(
    "<div class='vox-note'>"
    "<b>Continuous preview:</b> press Play as soon as audio becomes available. "
    "When you reach the generated edge, the player shows <b>Buffering — still generating</b>. "
    "When another chunk arrives, it extends the same preview and resumes near the same position."
    "</div>"
)

stream_player_output = widgets.Output()
chunk_preview_output = widgets.Output()
gen_output = widgets.Output()


def _stream_player_shell():
    return HTML(
        f"""
        <div class="vox-card" style="padding:14px;">
          <div style="font-weight:700;margin-bottom:8px;">
            🔴 Live combined preview
          </div>
          <audio
            id="{STREAM_PLAYER_ID}"
            controls
            preload="auto"
            style="width:100%;"
          ></audio>
          <div
            id="{STREAM_STATUS_ID}"
            style="margin-top:8px;color:#bdbdbd;font-size:12px;"
          >
            Waiting for the first completed chunk…
          </div>
        </div>
        """
    )


def _init_stream_player():
    stream_player_output.clear_output(wait=True)
    stream_player_output.append_display_data(
        _stream_player_shell()
    )

    js = Javascript(
        f"""
        (() => {{
          const id = {json.dumps(STREAM_PLAYER_ID)};
          const sid = {json.dumps(STREAM_STATUS_ID)};

          window.__voxcpmStreams = window.__voxcpmStreams || {{}};
          const s = window.__voxcpmStreams[id] = {{
            wantPlay: false,
            generationDone: false,
            availablePercent: 0,
            lastKnownTime: 0
          }};

          const getAudio = () => document.getElementById(id);
          const getStatus = () => document.getElementById(sid);

          const a = getAudio();
          if (!a) return;

          a.addEventListener("play", () => {{
            s.wantPlay = true;
            const t = getStatus();
            if (t) t.textContent =
              `Playing • ${{s.availablePercent}}% currently generated`;
          }});

          a.addEventListener("pause", () => {{
            s.lastKnownTime = Number.isFinite(a.currentTime) ? a.currentTime : 0;

            // When the user manually pauses before the generated edge,
            // do not force autoplay when the next chunk arrives.
            if (!a.ended && a.currentTime < Math.max(0, a.duration - 0.15)) {{
              s.wantPlay = false;
              const t = getStatus();
              if (t) t.textContent =
                `Paused • ${{s.availablePercent}}% currently generated`;
            }}
          }});

          a.addEventListener("timeupdate", () => {{
            s.lastKnownTime = Number.isFinite(a.currentTime) ? a.currentTime : 0;
          }});

          a.addEventListener("ended", () => {{
            s.lastKnownTime = Number.isFinite(a.duration) ? a.duration : s.lastKnownTime;

            const t = getStatus();

            if (!s.generationDone) {{
              // The listener reached the currently generated edge.
              // Remember that they want playback so the next update resumes.
              s.wantPlay = true;
              if (t) t.textContent =
                `⏳ Buffering — still generating… (${{s.availablePercent}}% ready)`;
            }} else {{
              s.wantPlay = false;
              if (t) t.textContent = "✓ Playback complete";
            }}
          }});
        }})();
        """
    )
    stream_player_output.append_display_data(js)


def _update_stream_player(data_url, percent, finished_chunks, total_chunks):
    """
    Replace the preview source with a longer cumulative file while preserving
    the listener's current position. If they were playing (or waiting at the
    generated edge), playback resumes automatically.
    """
    js = Javascript(
        f"""
        (() => {{
          const id = {json.dumps(STREAM_PLAYER_ID)};
          const sid = {json.dumps(STREAM_STATUS_ID)};
          const newSrc = {json.dumps(data_url)};
          const pct = {int(percent)};
          const finished = {int(finished_chunks)};
          const total = {int(total_chunks)};

          window.__voxcpmStreams = window.__voxcpmStreams || {{}};
          const s = window.__voxcpmStreams[id] || {{
            wantPlay: false,
            generationDone: false,
            availablePercent: 0,
            lastKnownTime: 0
          }};
          window.__voxcpmStreams[id] = s;

          const a = document.getElementById(id);
          const t = document.getElementById(sid);
          if (!a) return;

          const oldTime =
            Number.isFinite(a.currentTime) && a.currentTime > 0
              ? a.currentTime
              : (s.lastKnownTime || 0);

          const shouldResume = (!a.paused) || s.wantPlay;

          s.availablePercent = pct;
          s.lastKnownTime = oldTime;

          a.src = newSrc;
          a.load();

          const restore = async () => {{
            const duration = Number.isFinite(a.duration) ? a.duration : 0;
            const seekTo = Math.max(
              0,
              Math.min(oldTime, Math.max(0, duration - 0.05))
            );

            try {{
              if (seekTo > 0) a.currentTime = seekTo;
            }} catch (_) {{}}

            if (t) {{
              t.textContent =
                `${{pct}}% ready • ${{finished}}/${{total}} chunks generated`;
            }}

            if (shouldResume) {{
              s.wantPlay = true;
              try {{
                await a.play();
                if (t) t.textContent =
                  `Playing • ${{pct}}% ready • ${{finished}}/${{total}} chunks`;
              }} catch (_) {{
                if (t) t.textContent =
                  `${{pct}}% ready • press Play to continue`;
              }}
            }}
          }};

          a.addEventListener("loadedmetadata", restore, {{ once: true }});
        }})();
        """
    )

    stream_player_output.append_display_data(js)


def _finish_stream_player():
    js = Javascript(
        f"""
        (() => {{
          const id = {json.dumps(STREAM_PLAYER_ID)};
          const sid = {json.dumps(STREAM_STATUS_ID)};
          const streams = window.__voxcpmStreams || {{}};
          const s = streams[id];
          const t = document.getElementById(sid);

          if (s) {{
            s.generationDone = true;
            s.availablePercent = 100;
          }}

          if (t) {{
            t.textContent = "✓ 100% generated • complete file ready";
          }}
        }})();
        """
    )
    stream_player_output.append_display_data(js)


def _fail_stream_player(message):
    js = Javascript(
        f"""
        (() => {{
          const sid = {json.dumps(STREAM_STATUS_ID)};
          const t = document.getElementById(sid);
          if (t) t.textContent = {json.dumps("Generation failed: " + str(message))};
        }})();
        """
    )
    stream_player_output.append_display_data(js)


def build_delivery_control():
    base = DELIVERY_PRESETS.get(delivery_preset.value, "").strip()
    custom = custom_style.value.strip()
    return ", ".join([x for x in (base, custom) if x])


def make_generation_kwargs(chunk, ref_path, reference_text, control, seed):
    common = {
        "cfg_value": float(cfg_slider.value),
        "inference_timesteps": int(steps_slider.value),
        "retry_badcase": True,
        "retry_badcase_max_times": 3,
        "normalize": True,
    }

    # Only add a seed when deterministic mode is requested.
    # The worker handles older VoxCPM versions that do not expose seed=.
    if seed is not None:
        common["seed"] = int(seed)

    if control:
        # FIX FOR THE SPOKEN STYLE BUG:
        # controllable clone = reference audio only + (control)target_text.
        # Do NOT mix prompt_text/prompt_wav with the style instruction.
        common.update({
            "text": f"({control}){chunk}",
            "reference_wav_path": str(ref_path),
        })
    else:
        # Strong combined clone when no delivery instruction is requested.
        common.update({"text": chunk, "reference_wav_path": str(ref_path)})
        if reference_text:
            common.update({"prompt_wav_path": str(ref_path), "prompt_text": reference_text})

    return common


def _generate_long_job():
    try:
        vid = selected_voice_id()

        if not vid:
            raise ValueError("Save/select a permanent voice first.")

        script = gen_script.value.strip()

        if not script:
            raise ValueError("Paste a script first.")

        entry = library["voices"][vid]
        ref = voice_path(vid)

        if not ref.exists():
            raise FileNotFoundError("Selected reference WAV is missing.")

        reference_text = (
            entry.get("transcript", "")
            or ""
        ).strip()

        chunks = split_long_text(
            script,
            max_chars=int(chunk_chars.value),
        )

        if not chunks:
            raise ValueError("No chunks were created.")

        control = build_delivery_control()
        stamp = time.strftime("%Y%m%d_%H%M%S")

        batch_dir = GENERATED_DIR / (
            f"batch_{stamp}_{uuid.uuid4().hex[:6]}"
        )
        batch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        stream_dir = batch_dir / "stream_ready"
        stream_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        preview_wav = batch_dir / "live_preview.wav"
        preview_mp3 = batch_dir / "live_preview.mp3"

        # Reset all generation UI.
        gen_progress.min = 0
        gen_progress.max = len(chunks)
        gen_progress.value = 0
        gen_progress.bar_style = "info"

        gen_output.clear_output(wait=True)
        chunk_preview_output.clear_output(wait=True)
        _init_stream_player()

        gen_status.value = (
            f"<b>Generating…</b> 0/{len(chunks)} chunks • "
            f"two T4 workers • {entry.get('name', vid)}"
        )

        tasks = []
        base_seed = int(seed_box.value)

        for i, chunk in enumerate(chunks):
            seed = (
                None
                if random_seed_checkbox.value
                else base_seed + i
            )

            out = batch_dir / f"chunk_{i:04d}.wav"

            tasks.append({
                "chunk_index": i,
                "output_path": str(out),
                "kwargs": make_generation_kwargs(
                    chunk,
                    ref,
                    reference_text,
                    control,
                    seed,
                ),
            })

        # Two GPUs finish out of order. The preview must remain in script order.
        arrived = {}
        prepared = {}
        next_stream_index = 0
        callback_lock = threading.Lock()

        def on_chunk_finished(msg, finished_count, total_count):
            nonlocal next_stream_index

            idx = int(msg["chunk_index"])

            with callback_lock:
                arrived[idx] = msg
                gen_progress.value = int(finished_count)

                gen_status.value = (
                    f"<b>Generating…</b> {finished_count}/{total_count} "
                    f"worker tasks finished • building ordered stream"
                )

                old_next = next_stream_index

                # Reveal only the continuous prefix: 0,1,2,...
                while next_stream_index in arrived:
                    ready = arrived[next_stream_index]
                    part_path = (
                        stream_dir
                        / f"part_{next_stream_index:04d}.wav"
                    )

                    _, sr, sample_count = prepare_stream_chunk(
                        ready["output_path"],
                        part_path,
                        speed=float(speed_slider.value),
                        trim_edges=bool(trim_edges_checkbox.value),
                        trim_db=float(trim_db.value),
                    )

                    prepared[next_stream_index] = str(part_path)

                    part_seconds = (
                        float(sample_count) / float(sr)
                        if sr
                        else 0.0
                    )

                    # Always expose the individual chunk as a fallback/player.
                    chunk_preview_output.append_stdout(
                        "\n"
                        f"▶ Chunk {next_stream_index + 1}/{total_count} ready "
                        f"• {part_seconds:.1f}s • GPU {ready.get('gpu')}\n"
                    )

                    chunk_preview_output.append_display_data(
                        Audio(
                            filename=str(part_path),
                            autoplay=False,
                        )
                    )

                    next_stream_index += 1

                # Only rebuild the continuous stream when the ordered prefix grew.
                if next_stream_index > old_next:
                    ordered_ready = [
                        prepared[i]
                        for i in range(next_stream_index)
                    ]

                    make_stream_preview_mp3(
                        ordered_ready,
                        preview_wav,
                        preview_mp3,
                        pause_ms=int(join_pause.value),
                    )

                    pct = int(
                        round(
                            100.0
                            * next_stream_index
                            / total_count
                        )
                    )

                    data_url = file_as_data_url(
                        preview_mp3,
                        mime="audio/mpeg",
                    )

                    _update_stream_player(
                        data_url=data_url,
                        percent=pct,
                        finished_chunks=next_stream_index,
                        total_chunks=total_count,
                    )

        start = time.time()

        results = _VOX_SERVICE.run(
            tasks,
            on_result=on_chunk_finished,
        )

        # Fallback: prepare anything not handled by the preview callback.
        for result in results:
            idx = int(result["chunk_index"])

            if idx not in prepared:
                part_path = (
                    stream_dir / f"part_{idx:04d}.wav"
                )

                prepare_stream_chunk(
                    result["output_path"],
                    part_path,
                    speed=float(speed_slider.value),
                    trim_edges=bool(trim_edges_checkbox.value),
                    trim_db=float(trim_db.value),
                )

                prepared[idx] = str(part_path)

        ordered_prepared = [
            prepared[i]
            for i in range(len(chunks))
        ]

        final_path = (
            GENERATED_DIR
            / f"voiceover_{stamp}.wav"
        )

        _, sr, samples = concatenate_prepared_chunks(
            ordered_prepared,
            str(final_path),
            pause_ms=int(join_pause.value),
        )

        elapsed = time.time() - start
        duration = samples / sr
        used = sorted({
            r["gpu"]
            for r in results
        })

        gen_progress.value = len(chunks)
        gen_progress.bar_style = "success"

        gen_status.value = (
            f"<b>✓ Finished</b> • {elapsed:.1f}s processing • "
            f"{duration:.1f}s audio • GPUs {used}"
        )

        _finish_stream_player()

        # Final combined WAV must always appear in the interface.
        gen_output.clear_output(wait=True)
        gen_output.append_stdout(
            "\n✓ Final combined voiceover\n"
            f"{final_path}\n\n"
        )

        gen_output.append_display_data(
            Audio(
                filename=str(final_path),
                autoplay=False,
            )
        )

        gen_output.append_display_data(
            FileLink(
                str(final_path),
                result_html_prefix="📥 Download WAV: ",
            )
        )

    except Exception as e:
        gen_progress.bar_style = "danger"
        _fail_stream_player(str(e))

        gen_status.value = (
            "<span style='color:#ff7373'>"
            f"<b>Generation failed:</b> {str(e)}"
            "</span>"
        )

    finally:
        STATE["generation_busy"] = False
        generate_btn.disabled = False

def on_generate(_):
    if STATE["generation_busy"]:
        return
    STATE["generation_busy"] = True
    generate_btn.disabled = True
    threading.Thread(target=_generate_long_job, daemon=True).start()

generate_btn.on_click(on_generate)

generate_advanced = widgets.Accordion(children=[widgets.VBox([
    chunk_chars, join_pause, trim_edges_checkbox, trim_db, cfg_slider, steps_slider,
    widgets.HBox([seed_box, random_seed_checkbox]),
])], selected_index=None)
generate_advanced.set_title(0, "Advanced generation / chunk settings")

generate_tab = widgets.VBox([
    widgets.HTML('<div class="vox-section">Voice</div>'),
    widgets.HBox([saved_voice, play_selected_btn]),
    widgets.HTML('<div class="vox-section">Script</div>'),
    gen_script,
    widgets.HTML('<div class="vox-section">Voice settings</div>'),
    delivery_preset,
    custom_style,
    speed_slider,
    widgets.HTML(
        '<div class="vox-note"><b>Use Speed for reliable pace control.</b> '
        'Delivery/Custom are tone/emotion controls.</div>'
    ),
    generate_advanced,
    generate_btn,
    gen_status,
    gen_progress,
    gen_stream_summary,
    stream_player_output,
    widgets.HTML(
        '<div class="vox-section">Completed chunks</div>'
    ),
    widgets.HTML(
        '<div class="vox-note">'
        'Each finished chunk also appears below as an individual player. '
        'These are useful if the browser blocks automatic resume of the '
        'continuous preview.'
        '</div>'
    ),
    chunk_preview_output,
    widgets.HTML(
        '<div class="vox-section">Final combined file</div>'
    ),
    gen_output,
])


# ---------------------------
# Voice Design tab
# ---------------------------

profile_select = widgets.Dropdown(options=[(p["name"], pid) for pid, p in VOICE_PROFILES.items()], value="u_aung", description="Profile", layout=widgets.Layout(width="620px"))
profile_info = widgets.HTML()
design_audition_text = widgets.Textarea(
    value="မင်္ဂလာပါ။ ဒီအသံက သဘာဝကျပြီး နားထောင်ရလွယ်ကူတဲ့ မြန်မာစကားပြောသံ ဖြစ်စေချင်ပါတယ်။",
    placeholder="Use one to three sentences...", layout=widgets.Layout(width="100%", height="120px")
)
design_name = widgets.Text(value="My Burmese Voice", description="Save name", layout=widgets.Layout(width="620px"))
design_btn = widgets.Button(description="🎲 Generate profile candidate", button_style="success", layout=widgets.Layout(width="270px", height="46px"))
save_design_btn = widgets.Button(description="🔒 Save candidate", button_style="warning", disabled=True, layout=widgets.Layout(width="220px", height="46px"))
design_status = widgets.HTML()
design_output = widgets.Output()


def refresh_profile(change=None):
    profile = VOICE_PROFILES[profile_select.value]
    profile_info.value = f"<div class='vox-card'><b>{profile['name']}</b><br><span class='vox-note'>{profile['design']}</span></div>"

profile_select.observe(refresh_profile, names="value")
refresh_profile()


def _audition_job():
    try:
        pid = profile_select.value
        profile = VOICE_PROFILES[pid]
        text = design_audition_text.value.strip()
        if not text:
            raise ValueError("Enter a short audition script.")
        direction = profile["design"]
        out = GENERATED_DIR / f"audition_{pid}_{int(time.time())}_{uuid.uuid4().hex[:5]}.wav"
        design_status.value = f"<b>Generating:</b> {profile['name']}"
        result = _VOX_SERVICE.run([{
            "chunk_index": 0,
            "output_path": str(out),
            "kwargs": {
                "text": f"({direction}){text}",
                "cfg_value": 2.0,
                "inference_timesteps": 10,
                "retry_badcase": True,
                "normalize": True,
            },
        }])[0]
        STATE["audition_path"] = str(out)
        STATE["audition_text"] = text
        STATE["audition_direction"] = direction
        STATE["audition_profile_id"] = pid
        design_status.value = f"<b>✓ Candidate ready</b> • GPU {result['gpu']}"
        design_output.clear_output(wait=True)
        design_output.append_stdout(
            f"✓ Test voice ready: {out}\n\n"
        )
        design_output.append_display_data(
            Audio(
                filename=str(out),
                autoplay=False,
            )
        )
        design_output.append_display_data(
            FileLink(
                str(out),
                result_html_prefix="📥 Download test WAV: ",
            )
        )
        save_design_btn.disabled = False
    except Exception as e:
        design_status.value = f"<span style='color:#ff7373'><b>Failed:</b> {str(e)}</span>"
    finally:
        STATE["audition_busy"] = False
        design_btn.disabled = False


def on_design(_):
    if STATE["audition_busy"]:
        return
    STATE["audition_busy"] = True
    design_btn.disabled = True
    save_design_btn.disabled = True
    threading.Thread(target=_audition_job, daemon=True).start()


def on_save_design(_):
    candidate = STATE.get("audition_path")
    if not candidate or not Path(candidate).exists():
        design_status.value = "<span style='color:#ff7373'>Generate a candidate first.</span>"
        return
    name = design_name.value.strip() or "Designed Voice"
    vid = uuid.uuid4().hex[:12]
    dest = VOICE_DIR / f"{vid}.wav"
    shutil.copy2(candidate, dest)
    library["voices"][vid] = {
        "name": name,
        "file": f"voices/{vid}.wav",
        "transcript": STATE.get("audition_text") or "",
        "direction": STATE.get("audition_direction") or "",
        "profile_id": STATE.get("audition_profile_id"),
        "source_type": "voice_design",
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    library["active_voice_id"] = vid
    save_library()
    refresh_saved_voices(select_id=vid)
    design_status.value = f"<b>🔒 Saved:</b> {name}"


design_btn.on_click(on_design)
save_design_btn.on_click(on_save_design)

design_tab = widgets.VBox([
    widgets.HTML('<div class="vox-section">Choose one of 12 test voice profiles</div>'),
    profile_select, profile_info,
    widgets.HTML('<div class="vox-section">Short audition text</div>'),
    design_audition_text, design_name,
    widgets.HBox([design_btn, save_design_btn]),
    design_status, design_output,
])


# ---------------------------
# Voice Clone tab
# ---------------------------

clone_source_mode = widgets.ToggleButtons(options=[("Upload file", "upload"), ("Remote URL", "url")], value="upload", description="Source")
clone_upload = widgets.FileUpload(
    accept="audio/*,video/*,.wav,.mp3,.m4a,.aac,.flac,.ogg,.opus,.wma,.mp4,.mkv,.webm,.mov,.avi,.mpeg,.mpg,.m4v",
    multiple=False, description="Choose audio/video"
)
clone_url = widgets.Text(value="", placeholder="YouTube / TikTok / Google Drive / MEGA / direct URL", layout=widgets.Layout(width="100%"))
cookie_upload = widgets.FileUpload(accept=".txt", multiple=False, description="cookies.txt")
save_cookie_btn = widgets.Button(description="Use cookie file", layout=widgets.Layout(width="150px"))
clear_cookie_btn = widgets.Button(description="Clear cookies", layout=widgets.Layout(width="125px"))
cookie_status = widgets.HTML()
remote_user_agent = widgets.Text(value="", description="User-Agent", placeholder="optional", layout=widgets.Layout(width="100%"))
remote_referer = widgets.Text(value="", description="Referer", placeholder="optional", layout=widgets.Layout(width="100%"))
extract_vocals = widgets.Checkbox(value=False, description="Remove music / SFX with Demucs vocal isolation", indent=False)
reduce_noise = widgets.Checkbox(value=True, description="Reduce residual background noise", indent=False)
noise_amount = widgets.FloatSlider(value=0.70, min=0.20, max=1.00, step=0.05, description="Noise remove", continuous_update=False, layout=widgets.Layout(width="520px"))
clone_start = widgets.FloatText(value=0.0, description="Start sec", layout=widgets.Layout(width="260px"))
clone_max = widgets.FloatSlider(value=20.0, min=5.0, max=45.0, step=1.0, description="Max seconds", continuous_update=False, layout=widgets.Layout(width="520px"))
whisper_model = widgets.Dropdown(options=[("Whisper large-v3-turbo (faster)", "large-v3-turbo"), ("Whisper large-v3 (full)", "large-v3")], value="large-v3-turbo", description="ASR model", layout=widgets.Layout(width="620px"))
whisper_language = widgets.Dropdown(options=[("Burmese / Myanmar", "my"), ("Auto-detect", "auto"), ("English", "en")], value="my", description="Language", layout=widgets.Layout(width="500px"))
clone_name = widgets.Text(value="My Cloned Voice", description="Voice name", layout=widgets.Layout(width="620px"))
process_clone_btn = widgets.Button(description="✨ Prepare & transcribe clone", button_style="success", layout=widgets.Layout(width="280px", height="48px"))
save_clone_btn = widgets.Button(description="🔒 Save clone to library", button_style="warning", disabled=True, layout=widgets.Layout(width="250px", height="48px"))
clone_status = widgets.HTML()
clone_transcript = widgets.Textarea(value="", placeholder="Whisper transcript appears here; correct it if needed before saving.", layout=widgets.Layout(width="100%", height="150px"))
clone_output = widgets.Output()


def on_save_cookie(_):
    try:
        path = save_cookie_upload(cookie_upload)
        if not path:
            raise ValueError("Choose a Netscape cookies.txt file first.")
        STATE["cookie_file"] = path
        cookie_status.value = "<span style='color:#7ee787'>✓ Cookie file loaded for this session.</span>"
    except Exception as e:
        cookie_status.value = f"<span style='color:#ff7373'>{str(e)}</span>"


def on_clear_cookie(_):
    STATE["cookie_file"] = None
    p = COOKIE_DIR / "cookies.txt"
    if p.exists():
        try:
            p.unlink()
        except Exception:
            pass
    cookie_status.value = "Cookies cleared."

save_cookie_btn.on_click(on_save_cookie)
clear_cookie_btn.on_click(on_clear_cookie)


def set_clone_status(text, color="#d0d0d0"):
    clone_status.value = f"<div class='vox-card' style='color:{color}'>{text}</div>"


def _clone_pipeline():
    job_dir = IMPORT_DIR / f"clone_{int(time.time())}_{uuid.uuid4().hex[:6]}"
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        if clone_source_mode.value == "upload":
            set_clone_status("1/5 • Reading uploaded media…")
            source = save_upload_to_dir(clone_upload, job_dir, fallback_name="source.bin")
        else:
            set_clone_status("1/5 • Downloading remote media…")
            source = download_remote_media(
                clone_url.value, job_dir,
                cookie_file=STATE.get("cookie_file"),
                user_agent=remote_user_agent.value,
                referer=remote_referer.value,
            )
        STATE["clone_source"] = str(source)

        set_clone_status("2/5 • Converting to mono WAV and selecting the reference segment…")
        normalized = job_dir / "reference_normalized.wav"
        normalize_media_to_wav(source, normalized, start_seconds=max(0.0, float(clone_start.value)), max_seconds=max(1.0, float(clone_max.value)))
        working = str(normalized)

        if extract_vocals.value:
            set_clone_status("3/5 • Removing music / SFX with Demucs on CPU… this can take a few minutes.")
            try:
                working = extract_vocals_demucs(working, job_dir)
            except Exception as e:
                set_clone_status(f"Demucs failed; continuing with normalized source. {str(e)}", color="#ffcc66")
                working = str(normalized)
        else:
            set_clone_status("3/5 • Music/SFX isolation skipped.")

        if reduce_noise.value:
            set_clone_status("4/5 • Reducing residual background noise…")
            denoised = job_dir / "denoised.wav"
            reduce_residual_noise(working, denoised, amount=float(noise_amount.value))
            working = str(denoised)

        final_ref = job_dir / "clone_reference.wav"
        normalize_media_to_wav(working, final_ref, start_seconds=0, max_seconds=max(1.0, float(clone_max.value)))
        wav, sr = sf.read(str(final_ref), dtype="float32", always_2d=False)
        wav = trim_edge_silence(wav, sr, threshold_db=-45.0, keep_ms=40)
        sf.write(str(final_ref), wav, sr, subtype="PCM_16")

        set_clone_status(f"5/5 • Transcribing with {whisper_model.value} on CPU… first use downloads the ASR model.")
        transcript, detected = transcribe_reference(final_ref, model_name=whisper_model.value, language=whisper_language.value)
        STATE["clone_path"] = str(final_ref)
        STATE["clone_transcript"] = transcript
        clone_transcript.value = transcript

        with clone_output:
            clear_output()
            print("Prepared reference:")
            display(Audio(str(final_ref), autoplay=False))
            print("Detected language:", detected or "unknown")

        save_clone_btn.disabled = False
        set_clone_status("✓ Clone reference is ready. Listen, correct the transcript if necessary, then save it.", color="#7ee787")
    except Exception as e:
        set_clone_status(f"<b>Clone preparation failed:</b> {str(e)}", color="#ff7373")
    finally:
        STATE["clone_busy"] = False
        process_clone_btn.disabled = False


def on_process_clone(_):
    if STATE["clone_busy"]:
        return
    STATE["clone_busy"] = True
    process_clone_btn.disabled = True
    save_clone_btn.disabled = True
    threading.Thread(target=_clone_pipeline, daemon=True).start()


def on_save_clone(_):
    ref = STATE.get("clone_path")
    if not ref or not Path(ref).exists():
        set_clone_status("Prepare a clone reference first.", color="#ff7373")
        return
    transcript = clone_transcript.value.strip()
    if not transcript:
        set_clone_status("Transcript is empty. Enter the exact words spoken in the reference before saving.", color="#ffcc66")
        return
    name = clone_name.value.strip() or "Cloned Voice"
    vid = uuid.uuid4().hex[:12]
    dest = VOICE_DIR / f"{vid}.wav"
    shutil.copy2(ref, dest)
    library["voices"][vid] = {
        "name": name,
        "file": f"voices/{vid}.wav",
        "transcript": transcript,
        "direction": "",
        "profile_id": None,
        "source_type": "voice_clone",
        "source": STATE.get("clone_source", ""),
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    library["active_voice_id"] = vid
    save_library()
    refresh_saved_voices(select_id=vid)
    set_clone_status(f"🔒 Saved cloned voice: {name}", color="#7ee787")

process_clone_btn.on_click(on_process_clone)
save_clone_btn.on_click(on_save_clone)

cookie_advanced = widgets.Accordion(children=[widgets.VBox([
    widgets.HTML("<div class='vox-note'>Optional for media you are authorized to access. Upload a Netscape-format <b>cookies.txt</b>. Keep it private.</div>"),
    widgets.HBox([cookie_upload, save_cookie_btn, clear_cookie_btn]),
    cookie_status, remote_user_agent, remote_referer,
])], selected_index=None)
cookie_advanced.set_title(0, "Advanced remote-download cookies / headers")

cleanup_advanced = widgets.Accordion(children=[widgets.VBox([
    extract_vocals, reduce_noise, noise_amount, clone_start, clone_max, whisper_model, whisper_language,
])], selected_index=None)
cleanup_advanced.set_title(0, "Cleanup / transcription settings")

clone_tab = widgets.VBox([
    widgets.HTML('<div class="vox-section">Clone a voice from audio/video</div>'),
    widgets.HTML('<div class="vox-note">Use only audio you have permission to download and clone. Clean single-speaker audio gives the best result. Preprocessing runs in a background thread and uses CPU so it does not collide with the two T4 VoxCPM workers.</div>'),
    clone_source_mode,
    widgets.HTML("<b>Direct upload</b>"), clone_upload,
    widgets.HTML("<b>Remote URL</b>"), clone_url,
    cookie_advanced, cleanup_advanced, clone_name,
    widgets.HBox([process_clone_btn, save_clone_btn]),
    clone_status,
    widgets.HTML('<div class="vox-section">Reference transcript</div>'),
    clone_transcript, clone_output,
])


# ---------------------------
# Library / persistence tab
# ---------------------------

dataset_handle = widgets.Text(value=VOICE_DATASET_HANDLE_DEFAULT, description="Dataset", placeholder="username/dataset-slug", layout=widgets.Layout(width="620px"))
sync_btn = widgets.Button(description="☁ Sync voice library", button_style="info", layout=widgets.Layout(width="220px", height="44px"))
sync_status = widgets.HTML()


def on_sync(_):
    sync_btn.disabled = True
    try:
        sync_status.value = "Uploading current library…"
        sync_library_to_dataset(dataset_handle.value)
        sync_status.value = "<span style='color:#7ee787'>✓ Kaggle Dataset synced. Attach/use its latest version as a notebook Input after a fresh restart.</span>"
    except Exception as e:
        sync_status.value = f"<span style='color:#ff7373'>Sync failed: {str(e)}</span>"
    finally:
        sync_btn.disabled = False

sync_btn.on_click(on_sync)

library_tab = widgets.VBox([
    widgets.HTML('<div class="vox-section">Permanent voice library</div>'),
    widgets.HBox([saved_voice, play_selected_btn, delete_selected_btn]),
    library_output,
    widgets.HTML('<div class="vox-section">Persistent Kaggle Dataset</div>'),
    widgets.HTML('<div class="vox-note">Sync the working library to a Kaggle Dataset, then attach the latest Dataset version as an Input in future sessions. This notebook automatically restores library.json + voices/.</div>'),
    dataset_handle, sync_btn, sync_status,
])


# ---------------------------
# Render
# ---------------------------

refresh_saved_voices()
if restored_from:
    sync_status.value = f"<span style='color:#7ee787'>✓ Restored voice library from: {restored_from}</span>"

tabs = widgets.Tab(children=[generate_tab, design_tab, clone_tab, library_tab], layout=widgets.Layout(width="100%"))
tabs.set_title(0, "⚡ Generate")
tabs.set_title(1, "🎭 Voice Design")
tabs.set_title(2, "🧬 Voice Clone")
tabs.set_title(3, "💾 Library")

display(studio_css)
display(header)
display(tabs)


Loading one VoxCPM2 replica on each T4...
Starting 2 VoxCPM2 workers (optimize=False)...
  • launched worker PID 159 for physical GPU 0
  • launched worker PID 160 for physical GPU 1
  … still loading GPU worker(s) 0, 1 (0s elapsed)
  … still loading GPU worker(s) 0, 1 (10s elapsed)
Worker startup message: {'status': 'loading', 'gpu': 0}
Worker startup message: {'status': 'loading', 'gpu': 1}


voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False
voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Running on device: cuda, dtype: bfloat16
Running on device: cuda, dtype: bfloat16


  … still loading GPU worker(s) 0, 1 (20s elapsed)
  … still loading GPU worker(s) 0, 1 (30s elapsed)


Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors
Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors


  … still loading GPU worker(s) 0, 1 (40s elapsed)


Loaded VoxCPM2Model


✓ GPU 0 worker ready (48000 Hz)
✓ GPU 1 worker ready (48000 Hz)
✓ All VoxCPM2 GPU workers are ready.
✓ Dual-GPU VoxCPM2 service ready
✓ Runtime compatibility filter enabled for older/newer VoxCPM _generate signatures


Loaded VoxCPM2Model


HTML(value='\n<style>\n.vox-card {background:#181818;border:1px solid #343434;border-radius:14px;padding:16px …

HTML(value='\n<div class="vox-card">\n  <div class="vox-title">🎙️ VoxCPM2 Voice Studio</div>\n  <div class="vo…

## Notes

- **Style fix:** when a delivery instruction is present, the studio uses VoxCPM2 reference-audio-only controllable cloning instead of mixing the style instruction with prompt-text continuation. This prevents text such as “slightly faster” from being treated like narration.
- **Speed:** the Speed control uses FFmpeg `atempo`, so it is deterministic and preserves pitch.
- **Join pause:** generated edge silence is trimmed first; then the requested pause is inserted, so 100 ms/300 ms now has a predictable effect.
- **Clone preprocessing:** Demucs and Whisper run on CPU in a background thread because both T4s are reserved for VoxCPM2.
- Use only media you have permission to download and clone. Cookies stay in a separate private working folder and are never included by the voice-library sync. Imported source media and generated outputs are also excluded from the persistent Dataset.


### VoxCPM package compatibility
This revision inspects `model._generate` inside each GPU worker. Unsupported keyword arguments are removed automatically. On older builds without native `seed=`, a requested deterministic seed is applied directly to Python, NumPy and PyTorch RNGs instead.


### Progressive / live listening
Long generation now exposes completed chunks immediately. Because a normal WAV file is finalized with its length in the header and Kaggle/Jupyter rich output is served as a static object, the studio uses **ordered segmented streaming**: Part 1 appears as soon as it finishes, then Part 2, Part 3, and so on while both GPUs keep synthesizing later chunks. The final combined WAV is still created at the end.


### Streaming player v4
The live player now keeps one cumulative preview. After each **contiguous** chunk is ready, the notebook rebuilds a small 64-kbps MP3 preview and updates the same browser audio player while preserving its playback position. If playback reaches the currently generated edge first, the UI shows **“Buffering — still generating…”**; when another chunk arrives, the player attempts to resume automatically. Individual completed chunks are also shown as fallback audio players. The final downloadable result remains a full-quality WAV.
